# 🚀 TASK 6.3-C: CHIẾN DỊCH HUẤN LUYỆN THÍCH ỨNG ĐA MIỀN TRÊN GPU
### Huấn luyện 3 nhánh ứng viên M3 (+Data), M4 (+Distillation), M5 (+LoRA) — Milestone 6

> **Mã nhiệm vụ**: `Task 6.3-C` *(Milestone 6 - Tuần 6)*  
> **Người chủ trì**: **Thành viên C (Training & Evaluation Lead)**  
> **Người phối hợp**: **Thành viên B (Architecture Lead)**, **Thành viên A (Data & Infra Lead)**  
> **Môi trường khuyến nghị**: Kaggle Server (GPU RTX PRO 6000 Blackwell 95GB) hoặc Google Colab Pro (GPU Tesla T4 / A100)  
> **Kế thừa hạ tầng**: Kho dữ liệu và bộ lấy mẫu cân bằng Task 6.2; Hàm mất mát Distillation & Consistency Task 6.3-B; Khóa cố định 18 tập test Task 5.1  

---
### 📌 TIÊU CHUẨN NGHIỆM THU (DEFINITION OF DONE - DoD):
- [x] Đã vượt qua Smoke Gate Milestone 6 với 100% tiêu chí pass.
- [ ] Cố định điều kiện đối chứng cho toàn bộ 3 nhánh: cùng random seed (42), cùng data manifest, cùng khởi tạo trọng số, cùng learning rate schedule (5 epochs).
- [ ] Tuân thủ nghiêm ngặt chính sách đĩa: Chỉ lưu `checkpoint_latest.pth` và `crossdomain_model_best.pth`, không lưu dư thừa từng epoch.
- [ ] Thu thập đầy đủ bản ghi tiến trình qua từng epoch vào `DATASET/logs/m6_training_history.csv`.
- [ ] Checkpoint tuyển chọn của từng nhánh được sao lưu an toàn sang Google Drive 5TB.


# ==============================================================================
# 1. THIET LAP MOI TRUONG VA KIEM TRA GPU
# ==============================================================================
import os
import sys
import time
import torch

print(f"[*] Phien ban PyTorch: {torch.__version__}")
print(f"[*] CUDA kha dung: {torch.cuda.is_available()}")

if torch.cuda.is_available():
    gpu_name = torch.cuda.get_device_name(0)
    vram_gb = torch.cuda.get_device_properties(0).total_memory / (1024 ** 3)
    print(f"[*] GPU dang su dung: {gpu_name} ({vram_gb:.2f} GB VRAM)")
else:
    print("[!] Canh bao: Khong tim thay GPU, script se chay tren CPU (che do dry-run)!")

# Nhan dien nen tang
IS_COLAB = 'google.colab' in sys.modules
IS_KAGGLE = os.path.exists('/kaggle/working')

if IS_COLAB:
    print("[*] Phat hien moi truong Google Colab. Tien hanh mount Drive...")
    from google.colab import drive
    drive.mount('/content/drive')
    WORKSPACE = '/content/fatformer-xla'
    BRANCH = 'hoang'
    REPO_URL = 'https://github.com/LeeVHoangtk3/fatformer-xla.git'
    if not os.path.exists(WORKSPACE):
        print(f"[*] Dang clone ma nguon tu {REPO_URL} (nhanh {BRANCH})...")
        os.system(f"git clone -b {BRANCH} {REPO_URL} {WORKSPACE}")
    else:
        print(f"[*] Dong bo commit tu nhanh {BRANCH}...")
        os.system(f"git -C {WORKSPACE} fetch origin {BRANCH}")
        os.system(f"git -C {WORKSPACE} reset --hard origin/{BRANCH}")
    DRIVE_HUB = '/content/drive/MyDrive/Fatformer'
elif IS_KAGGLE:
    print("[*] Phat hien moi truong Kaggle Competition Server.")
    WORKSPACE = '/kaggle/working/fatformer-xla'
    DRIVE_HUB = '/kaggle/input'
else:
    print("[*] Phat hien moi truong Local Machine.")
    cur = os.path.abspath(os.getcwd())
    WORKSPACE = cur
    while cur and os.path.dirname(cur) != cur:
        if os.path.exists(os.path.join(cur, 'configs')) and os.path.exists(os.path.join(cur, 'src')):
            WORKSPACE = cur
            break
        cur = os.path.dirname(cur)
    DRIVE_HUB = os.path.join(WORKSPACE, 'DATASET')

if os.path.exists(WORKSPACE) and WORKSPACE not in sys.path:
    sys.path.insert(0, WORKSPACE)
print(f"[*] Workspace lam viec: {WORKSPACE}")


In [ ]:
# ==============================================================================
# 1. THIET LAP MOI TRUONG VA KIEM TRA GPU
# ==============================================================================
import os
import sys
import time
import torch

print(f"[*] Phien ban PyTorch: {torch.__version__}")
print(f"[*] CUDA kha dung: {torch.cuda.is_available()}")

if torch.cuda.is_available():
    gpu_name = torch.cuda.get_device_name(0)
    vram_gb = torch.cuda.get_device_properties(0).total_memory / (1024 ** 3)
    print(f"[*] GPU dang su dung: {gpu_name} ({vram_gb:.2f} GB VRAM)")
else:
    print("[!] Canh bao: Khong tim thay GPU, script se chay tren CPU (che do dry-run)!")

# Nhan dien nen tang
IS_COLAB = 'google.colab' in sys.modules
IS_KAGGLE = os.path.exists('/kaggle/working')

if IS_COLAB:
    print("[*] Phat hien moi truong Google Colab. Tien hanh mount Drive...")
    from google.colab import drive
    drive.mount('/content/drive')
    WORKSPACE = '/content/fatformer-xla'
    BRANCH = 'hoang'
    REPO_URL = 'https://github.com/LeeVHoangtk3/fatformer-xla.git'
    if not os.path.exists(WORKSPACE):
        print(f"[*] Dang clone ma nguon tu {REPO_URL} (nhanh {BRANCH})...")
        os.system(f"git clone -b {BRANCH} {REPO_URL} {WORKSPACE}")
    else:
        print(f"[*] Dong bo commit tu nhanh {BRANCH}...")
        os.system(f"git -C {WORKSPACE} pull origin {BRANCH}")
    DRIVE_HUB = '/content/drive/MyDrive/Fatformer'
elif IS_KAGGLE:
    print("[*] Phat hien moi truong Kaggle Competition Server.")
    WORKSPACE = '/kaggle/working/fatformer-xla'
    DRIVE_HUB = '/kaggle/input'
else:
    print("[*] Phat hien moi truong Local Machine.")
    # Tu dong do tim thu muc goc chua configs va src
    cur = os.path.abspath(os.getcwd())
    WORKSPACE = cur
    while cur and os.path.dirname(cur) != cur:
        if os.path.exists(os.path.join(cur, 'configs')) and os.path.exists(os.path.join(cur, 'src')):
            WORKSPACE = cur
            break
        cur = os.path.dirname(cur)
    DRIVE_HUB = os.path.join(WORKSPACE, 'DATASET')

if os.path.exists(WORKSPACE) and WORKSPACE not in sys.path:
    sys.path.insert(0, WORKSPACE)
print(f"[*] Workspace lam viec: {WORKSPACE}")


## 2. Kích Hoạt Kiểm Tra Khớp Nối Cổng Smoke Gate Milestone 6
Trước khi khởi động huấn luyện 5 epochs tiêu tốn tài nguyên, thực thi kiểm tra tích hợp 1 batch autograd:
* Đảm bảo dữ liệu sinh từ `BalancedCrossDomainSampler` (Task 6.2) nạp mượt mà vào `DataLoader`.
* Đảm bảo mô hình Teacher đóng băng 100%, không bị rò rỉ gradient.
* Đảm bảo hàm tổn thất `CrossDomainTotalLoss` (Task 6.3-B) tính toán ổn định, không có NaN/Inf.


In [ ]:
# ==============================================================================
# 4. HUAN LUYEN NHANH M3 — DATA ONLY (5 EPOCHS)
# ==============================================================================
train_script = os.path.join(WORKSPACE, 'src', 'training', 'train_crossdomain.py')
cfg_m3 = os.path.join(WORKSPACE, 'configs', 'train_m3_data.yaml')

device_flag = "--device cuda" if torch.cuda.is_available() else "--dry_run --device cpu"
cmd_m3 = f"python -u {train_script} --config {cfg_m3} {device_flag}"

print(f"[*] Thuc thi lenh: {cmd_m3}")
t0 = time.time()
ret_m3 = os.system(cmd_m3)
elapsed_m3 = time.time() - t0

if ret_m3 != 0:
    raise RuntimeError(f"[X] Huan luyen nhanh M3 that bai voi exit code: {ret_m3}! Vui long kiem tra log ben tren.")
else:
    print(f"[✓] Huan luyen nhanh M3 hoan tat thanh cong trong {elapsed_m3:.2f}s!")


## 3. Xác Nhận Kê Khai Dữ Liệu Đa Miền & Bộ Lấy Mẫu Cân Bằng
* Đọc tệp kê khai nguồn gốc `DATASET/manifests/data_manifest.json` đã nghiệm thu tại Task 6.2.
* Phân chia rõ ràng: `train` (huấn luyện), `val_calibration` (hiệu chuẩn nhiệt độ Task 6.4), `val_selection` (xếp hạng độc lập Task 6.4).
* Khóa cố định 100% không chạm vào 18 tập test của Task 5.1.


In [ ]:
# ==============================================================================
# 5. HUAN LUYEN NHANH M4 — DATA + DISTILLATION + CONSISTENCY (5 EPOCHS)
# ==============================================================================
cfg_m4 = os.path.join(WORKSPACE, 'configs', 'train_m4_distill.yaml')

device_flag = "--device cuda" if torch.cuda.is_available() else "--dry_run --device cpu"
cmd_m4 = f"python -u {train_script} --config {cfg_m4} {device_flag}"

print(f"[*] Thuc thi lenh: {cmd_m4}")
t0 = time.time()
ret_m4 = os.system(cmd_m4)
elapsed_m4 = time.time() - t0

if ret_m4 != 0:
    raise RuntimeError(f"[X] Huan luyen nhanh M4 that bai voi exit code: {ret_m4}! Vui long kiem tra log ben tren.")
else:
    print(f"[✓] Huan luyen nhanh M4 hoan tat thanh cong trong {elapsed_m4:.2f}s!")


## 4. Huấn Luyện Nhánh M3: Mở Rộng Dữ Liệu Đa Miền (Data Only)
* **Cấu hình**: `configs/train_m3_data.yaml`
* **Đặc trưng**: Mở rộng dữ liệu đa bộ sinh (ProGAN, StyleGAN, Stable Diffusion, Midjourney) với hàm mất mát Cross-Entropy tiêu chuẩn.
* **Mục tiêu đối chứng**: Xác định mức cải thiện chỉ nhờ mở rộng phân phối dữ liệu huấn luyện, chưa áp dụng cơ chế chống quên tri thức mỏ neo.


In [ ]:
# ==============================================================================
# 6. HUAN LUYEN NHANH M5 — M4 + LORA VIT ADAPTER (5 EPOCHS)
# ==============================================================================
cfg_m5 = os.path.join(WORKSPACE, 'configs', 'train_m5_lora.yaml')

device_flag = "--device cuda" if torch.cuda.is_available() else "--dry_run --device cpu"
cmd_m5 = f"python -u {train_script} --config {cfg_m5} {device_flag}"

print(f"[*] Thuc thi lenh: {cmd_m5}")
t0 = time.time()
ret_m5 = os.system(cmd_m5)
elapsed_m5 = time.time() - t0

if ret_m5 != 0:
    raise RuntimeError(f"[X] Huan luyen nhanh M5 that bai voi exit code: {ret_m5}! Vui long kiem tra log ben tren.")
else:
    print(f"[✓] Huan luyen nhanh M5 hoan tat thanh cong trong {elapsed_m5:.2f}s!")


## 5. Huấn Luyện Nhánh M4: Tích Hợp Distillation & Consistency Loss
* **Cấu hình**: `configs/train_m4_distill.yaml`
* **Đặc trưng**: Bổ sung hàm mất mát triệt tiêu phân kỳ KL Distillation ($lpha_{	ext{kd}}=0,4$) với mô hình Teacher mỏ neo đóng băng và ràng buộc đồng bộ phân phối Logits Consistency ($eta_{	ext{cons}}=0,2$) trên cặp ảnh sạch - suy biến.
* **Mục tiêu đối chứng**: Ngăn chặn hiện tượng quên thảm khốc trên ảnh sạch phòng lab, đồng thời bảo tồn tính bất biến phân loại dưới suy biến nén sâu.


In [ ]:
# ==============================================================================
# 5. HUAN LUYEN NHANH M4 — DATA + DISTILLATION + CONSISTENCY (5 EPOCHS)
# ==============================================================================
cfg_m4 = os.path.join(WORKSPACE, 'configs', 'train_m4_distill.yaml')

cmd_m4 = f"python {train_script} --config {cfg_m4}"
print(f"[*] Thuc thi lenh: {cmd_m4}")

if not torch.cuda.is_available():
    cmd_m4 += " --dry_run --device cpu"

t0 = time.time()
ret_m4 = os.system(cmd_m4)
print(f"[*] Nhieu vu nhanh M4 hoan tat trong {time.time() - t0:.2f}s (Exit code: {ret_m4})")


## 6. Huấn Luyện Nhánh M5: Bổ Sung Bộ Điều Hợp LoRA ViT Adapter
* **Cấu hình**: `configs/train_m5_lora.yaml`
* **Đặc trưng**: Kế thừa toàn bộ hàm mất mát của M4, đồng thời gắn thêm module LoRA ($r=8, lpha=16$) vào các khối phép chiếu tuyến tính $Q, V$ của CLIP ViT-L/14.
* **Mục tiêu đối chứng**: Mở rộng dung lượng biểu diễn vi mô của không gian đặc trưng thị giác mà không làm thay đổi các tầng trích xuất toàn cục đã đóng băng.


In [ ]:
# ==============================================================================
# 6. HUAN LUYEN NHANH M5 — M4 + LORA VIT ADAPTER (5 EPOCHS)
# ==============================================================================
cfg_m5 = os.path.join(WORKSPACE, 'configs', 'train_m5_lora.yaml')

cmd_m5 = f"python {train_script} --config {cfg_m5}"
print(f"[*] Thuc thi lenh: {cmd_m5}")

if not torch.cuda.is_available():
    cmd_m5 += " --dry_run --device cpu"

t0 = time.time()
ret_m5 = os.system(cmd_m5)
print(f"[*] Nhieu vu nhanh M5 hoan tat trong {time.time() - t0:.2f}s (Exit code: {ret_m5})")


## 7. Tổng Hợp Bản Ghi Tiến Trình & Đối Soát Sơ Bộ
* Đọc tệp bản ghi `DATASET/logs/m6_training_history.csv` ghi nhận chi tiết loss và validation metrics qua từng epoch của cả 3 nhánh.
* Vẽ biểu đồ trực quan hóa đường cong hội tụ.
* Đối chiếu sơ bộ với bảng mốc sàn baseline `val_crossdomain_baseline_metrics.json` từ Task 6.2.


In [ ]:
# ==============================================================================
# 7. TONG HOP BAN GHI TIEN TRINH VA VE BIEU DO DOI CHIEU
# ==============================================================================
import pandas as pd
import matplotlib.pyplot as plt

log_csv = os.path.join(WORKSPACE, 'DATASET', 'logs', 'm6_training_history.csv')
if os.path.exists(log_csv):
    df = pd.read_csv(log_csv)
    print("[*] Trich doan ban ghi tien trinh:")
    print(df.tail(10))
    
    # Ve bieu do so sanh Loss
    plt.figure(figsize=(12, 5))
    plt.subplot(1, 2, 1)
    for branch in df['branch'].unique():
        sub = df[df['branch'] == branch]
        plt.plot(sub['epoch'], sub['train_loss'], marker='o', label=f"{branch} Train Loss")
    plt.title("So Sanh Tien Trinh Giam Loss Qua Cac Epoch")
    plt.xlabel("Epoch")
    plt.ylabel("Loss")
    plt.legend()
    plt.grid(True, alpha=0.3)
    
    plt.subplot(1, 2, 2)
    for branch in df['branch'].unique():
        sub = df[df['branch'] == branch]
        plt.plot(sub['epoch'], sub['val_macro_acc'], marker='s', label=f"{branch} Val Macro ACC")
    plt.axhline(y=73.25, color='r', linestyle='--', label='Baseline Moc San (73.25%)')
    plt.title("So Sanh Do Chinh Xac Val Macro ACC")
    plt.xlabel("Epoch")
    plt.ylabel("Macro ACC (%)")
    plt.legend()
    plt.grid(True, alpha=0.3)
    plt.tight_layout()
    
    chart_path = os.path.join(WORKSPACE, 'DATASET', 'logs', 'm6_convergence_chart.png')
    plt.savefig(chart_path, dpi=150)
    print(f"[*] Bieu do hoi tu da luu tai: {chart_path}")
    plt.show()
else:
    print(f"[*] Ban ghi {log_csv} se duoc tao sau khi hoan tat chay tren GPU.")


## 8. Sao Lưu Checkpoint & Nhật Ký Lên Google Drive 5TB
* Tự động sao chép toàn bộ checkpoints (`m3`, `m4`, `m5`) và nhật ký huấn luyện `m6_training_history.csv` sang Google Drive `/content/drive/MyDrive/Fatformer/`.
* Đối soát trực quan kích thước và danh sách tệp trên Drive để phục vụ chuyển giao cho Task 6.4 và Task 6.5.


In [ ]:
# ==============================================================================
# 8. SAO LUU CHECKPOINTS VA NHAT KY SANG GOOGLE DRIVE 5TB
# ==============================================================================
import os
import shutil

src_ckpt = os.path.join(WORKSPACE, 'DATASET', 'checkpoints')
dst_ckpt = '/content/drive/MyDrive/Fatformer/checkpoints'
src_logs = os.path.join(WORKSPACE, 'DATASET', 'logs')
dst_logs = '/content/drive/MyDrive/Fatformer/logs'

print("[*] Tien hanh sao luu checkpoints sang Google Drive...")
if os.path.exists(src_ckpt):
    os.makedirs(dst_ckpt, exist_ok=True)
    os.system(f"cp -rv {src_ckpt}/* {dst_ckpt}/")
else:
    print(f"[!] Canh bao: Khong tim thay thu muc {src_ckpt}")

print("\n[*] Tien hanh sao luu nhat ky sang Google Drive...")
if os.path.exists(src_logs):
    os.makedirs(dst_logs, exist_ok=True)
    os.system(f"cp -rv {src_logs}/* {dst_logs}/")

# Kiem tra doi chieu danh sach tep tren Google Drive
print("\n[✓] Kiem tra cac tep tren Google Drive:")
if os.path.exists(dst_ckpt):
    for root, dirs, files in os.walk(dst_ckpt):
        for f in files:
            p = os.path.join(root, f)
            size_mb = os.path.getsize(p) / (1024 * 1024)
            print(f"  • {p} ({size_mb:.2f} MB)")


## 9. Bàn Giao Thành Phẩm Cho Task 6.5 & Task 6.4
1. **Bàn giao cho Task 6.5 (Ablation Study)**:
   - Toàn bộ kết quả định lượng của 3 nhánh M3, M4, M5 cùng nhánh đối chứng M1 (Plan A) và M2 (Aug only).
   - Lập ma trận Ablation Study đa chiều hoàn chỉnh.
2. **Bàn giao cho Task 6.4 (Hiệu Chuẩn & Tuyển Chọn Checkpoint)**:
   - Checkpoint đã lưu của từng nhánh trên Drive:
     - `/content/drive/MyDrive/Fatformer/checkpoints/m3/crossdomain_model_best.pth`
     - `/content/drive/MyDrive/Fatformer/checkpoints/m4/crossdomain_model_best.pth`
     - `/content/drive/MyDrive/Fatformer/checkpoints/m5/crossdomain_model_best.pth`
   - Script `tools/calibrate_and_select.py` sẽ nạp các checkpoint này để tối ưu hóa nhiệt độ $T^*$ trên `val_calibration` và tuyển chọn checkpoint vượt trội theo Gate 4 điều kiện trên `val_selection`.
